# Data Preprocessing Pipeline — TechJobAI

Notebook này thực hiện ETL từ raw Kaggle LinkedIn CSVs → file `it_jobs_processed.csv`
thành một pipeline thống nhất, có chú thích chi tiết cho người kế thừa.

## Mục tiêu

1. Đọc 3 file raw từ Kaggle LinkedIn Job Postings (1.3M bài đăng)
2. Lọc chỉ giữ lại các công việc IT (~129K)
3. Chuẩn hoá: cấp bậc, bang, lĩnh vực IT, loại hình làm việc
4. Trích xuất lương từ mô tả công việc bằng regex
5. Đếm kỹ năng theo 9 nhóm
6. Xuất file `data/it_jobs_processed.csv` (~27 MB)

## Kiến trúc dữ liệu

```
Dataset/                          data/
├── linkedin_job_postings.csv  ──►├── it_jobs_processed.csv
├── job_skills.csv                ├── backup_trends.csv
├── job_summary.csv               └── realtime_cache.json
└── preprocess_kaggle.py
```

---
## 1. Import thư viện

Yêu cầu: `pandas`, `numpy`, `re`, `csv`, `gc` (đều có trong requirements.txt)

In [ ]:
import os, re, gc, csv
import numpy as np
import pandas as pd

# ── Đường dẫn ──
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))  # Lên root project
DATA_DIR = os.path.join(BASE_DIR, "data")
DATASET_DIR = os.path.join(BASE_DIR, "Dataset")

# File raw từ Kaggle (cần tải về trước)
POSTINGS = os.path.join(DATASET_DIR, "linkedin_job_postings.csv")   # 396 MB
SKILLS   = os.path.join(DATASET_DIR, "job_skills.csv")              # 642 MB
SUMMARY  = os.path.join(DATASET_DIR, "job_summary.csv")            # 4.9 GB

# File đầu ra
OUTPUT   = os.path.join(DATA_DIR, "it_jobs_processed.csv")

CHUNK = 200_000   # Số dòng mỗi chunk khi đọc skills (tiết kiệm RAM)

---
## 2. Định nghĩa từ khoá

### 2a. Từ khoá IT — lọc công việc IT từ 1.3M bài đăng

Một bài đăng được coi là IT nếu `job_title` chứa **bất kỳ** từ nào dưới đây.

In [ ]:
IT_KW = [
    "software", "engineer", "developer", "data scien", "machine learning",
    "ai ", "devops", "cloud", "full stack", "frontend", "backend",
    "mobile", "security", "qa ", "test", "sre", "platform", "infrastructure",
    "python", "java", "javascript", "react", "node", "golang",
]

### 2b. Nhóm kỹ năng — 9 nhóm, mỗi nhóm là danh sách từ khoá

Dùng để đếm số kỹ năng trong `job_skills` + `job_summary`.

In [ ]:
SKILL_KW = {
    "programming": ["python", "java", "javascript", "typescript", "c++", "c#", "ruby", "golang", "rust", "php"],
    "cloud":       ["aws", "azure", "gcp", "google cloud", "terraform", "cloud"],
    "ai_ml":       ["machine learning", "deep learning", "artificial intelligence", "nlp", "computer vision", "tensorflow", "pytorch", "keras", "scikit"],
    "database":    ["sql", "mysql", "postgresql", "oracle", "mongodb", "redis", "elasticsearch"],
    "devops":      ["docker", "kubernetes", "jenkins", "ci/cd", "github actions", "ansible"],
    "framework":   ["react", "angular", "vue", "next.js", "node.js", "django", "flask", "fastapi", "spring", ".net"],
    "data_engineering": ["apache spark", "hadoop", "kafka", "airflow", "etl"],
    "security":    ["cybersecurity", "penetration testing", "vulnerability", "encryption", "firewall"],
    "soft_skills": ["communication", "leadership", "teamwork", "agile", "scrum"],
}

### 2c. Regex trích xuất lương

Hai pattern phổ biến trong job description:
- `$120K` hoặc `$120k`
- `$120,000`

In [ ]:
SALARY_RE = re.compile(r'\$([0-9]{2,3})[kK]\b|\$([0-9]{2,3}),[0-9]{3}\b', re.IGNORECASE)

---
## 3. Hàm chuẩn hoá

### 3a. `ns(loc)` — Chuẩn hoá địa điểm → mã bang US

Input có thể là:
- `"CA"` → giữ nguyên
- `"San Francisco, CA"` → trích `"CA"`
- `"california"` → `"CA"` (full name)
- `"ca"` → `"CA"` (lowercase abbrev)
- Không xác định → `"Remote"`

In [ ]:
_STATE_MAP = {
    "alabama": "AL", "alaska": "AK", "arizona": "AZ", "arkansas": "AR",
    "california": "CA", "colorado": "CO", "connecticut": "CT", "delaware": "DE",
    "florida": "FL", "georgia": "GA", "hawaii": "HI", "idaho": "ID",
    "illinois": "IL", "indiana": "IN", "iowa": "IA", "kansas": "KS",
    "kentucky": "KY", "louisiana": "LA", "maine": "ME", "maryland": "MD",
    "massachusetts": "MA", "michigan": "MI", "minnesota": "MN", "mississippi": "MS",
    "missouri": "MO", "montana": "MT", "nebraska": "NE", "nevada": "NV",
    "new hampshire": "NH", "new jersey": "NJ", "new mexico": "NM", "new york": "NY",
    "north carolina": "NC", "north dakota": "ND", "ohio": "OH", "oklahoma": "OK",
    "oregon": "OR", "pennsylvania": "PA", "rhode island": "RI", "south carolina": "SC",
    "south dakota": "SD", "tennessee": "TN", "texas": "TX", "utah": "UT",
    "vermont": "VT", "virginia": "VA", "washington": "WA", "west virginia": "WV",
    "wisconsin": "WI", "wyoming": "WY", "district of columbia": "DC",
}
_ABBREV_MAP = {v.lower(): v for v in _STATE_MAP.values()}  # "ca" → "CA"

def ns(loc):
    """Normalise location string to US state abbreviation."""
    if not loc:
        return "Unknown"
    loc = str(loc).strip()
    # Đã là mã 2 chữ in hoa
    if len(loc) <= 3 and loc.isalpha() and loc.isupper():
        return loc
    # Lấy phần cuối sau dấu phẩy: "San Francisco, CA" → "CA"
    tail = loc.split(",")[-1].strip().lower()
    if tail in _STATE_MAP:
        return _STATE_MAP[tail]
    if tail in _ABBREV_MAP:
        return _ABBREV_MAP[tail]
    # "california" → "CA"
    if loc.lower() in _STATE_MAP:
        return _STATE_MAP[loc.lower()]
    return "Remote"

### 3b. `nsen(level, title)` — Xác định cấp bậc

**Chiến lược:**
1. Ưu tiên trích từ `job_title` (vì `job_level` của Kaggle chỉ có 2 giá trị: "Mid senior" và "Associate")
2. Nếu title không có tín hiệu → fallback sang `job_level`

4 cấp bậc: `Junior`, `Mid`, `Senior`, `Manager`

In [ ]:
def nsen(level, title):
    """Xác định seniority level từ job_title (ưu tiên) + job_level (fallback)."""
    # ── Bước 1: Trích từ job_title ──
    t = str(title).lower() if title else ""
    if any(k in t for k in ("intern", "internship", "entry level", "entry-level",
                            "junior", "jr", "graduate", "trainee", "apprentice")):
        return "Junior"
    if any(k in t for k in ("manager", "director", "head of", "vp ")):
        return "Manager"
    if any(k in t for k in ("senior", "sr ", "lead", "principal", "staff",
                            "architect", "expert", "vp of", "head", "fellow")):
        return "Senior"

    # ── Bước 2: Fallback sang job_level (Kaggle) ──
    v = str(level).lower().strip() if level else ""
    if any(k in v for k in ("intern", "entry", "junior", "graduate")):
        return "Junior"
    if "associate" in v:          # Associate là entry-level
        return "Junior"
    if any(k in v for k in ("manager", "director", "head", "vp")):
        return "Manager"
    if "mid" in v:                # "Mid senior" → Mid
        return "Mid"
    if any(k in v for k in ("senior", "lead", "principal", "staff", "executive")):
        return "Senior"

    return "Mid"  # Mặc định

### 3c. `md(title)` — Xác định lĩnh vực IT

5 domain: `Software Engineering`, `DevOps/SRE`, `Cybersecurity`, `Data Science`, `QA/Testing`

In [ ]:
def md(title):
    """Map job_title → IT domain."""
    t = str(title).lower()
    if any(k in t for k in ["data", "ai", "machine learning", "nlp"]):
        return "Data Science"
    if any(k in t for k in ["cloud", "devops", "sre", "system", "network", "infrastructure"]):
        return "DevOps/SRE"
    if any(k in t for k in ["security", "cyber"]):
        return "Cybersecurity"
    if any(k in t for k in ["test", "qa", "quality"]):
        return "QA/Testing"
    return "Software Engineering"

### 3d. `njt(v)` — Loại hình làm việc

In [ ]:
def njt(v):
    """Normalise job type."""
    v = str(v).lower().strip() if v else ""
    if "remote" in v: return "Remote"
    if "hybrid" in v: return "Hybrid"
    if "full" in v: return "Full-time"
    if "part" in v: return "Part-time"
    if "contract" in v: return "Contract"
    return "Onsite" if v else "Unknown"

### 3e. `count_skill(text, kws)` — Đếm kỹ năng

Dùng `re.search` với word boundary để tránh false positive (vd: "c++" không match "javascript").

In [ ]:
def count_skill(text, kws):
    """Đếm số từ khoá kỹ năng xuất hiện trong text."""
    if not text:
        return 0
    t = str(text).lower()
    return sum(1 for kw in kws if re.search(r'\b' + re.escape(kw.strip()) + r'\b', t))

### 3f. `extract_salary(text)` — Trích lương từ job summary

Dùng regex tìm `$XXK` hoặc `$XX,XXX`. Nếu có nhiều mức thì lấy trung bình.

In [ ]:
def extract_salary(text):
    """Trích xuất salary từ job summary bằng regex."""
    if not text:
        return np.nan
    m = SALARY_RE.findall(str(text))
    if m:
        vals = [float(a)*1000 if a else float(b)*1000 for a, b in m if a or b]
        return np.mean(vals) if vals else np.nan
    return np.nan

---
## 4. Pipeline chính

### Bước 1: Đọc postings (`linkedin_job_postings.csv`)

Chỉ lấy các cột cần thiết để tiết kiệm RAM.

In [ ]:
print("[1/4] Loading postings...")
df_post = pd.read_csv(
    POSTINGS,
    usecols=["job_link", "job_title", "company", "job_location", "job_level", "job_type"],
    low_memory=False,
)
print(f"  {len(df_post)} rows")

# Lọc IT jobs
title_lower = df_post["job_title"].str.lower()
it_mask = title_lower.apply(lambda t: any(k in t for k in IT_KW) if pd.notna(t) else False)
df_it = df_post[it_mask].copy()
del df_post; gc.collect()
it_links = set(df_it["job_link"].values)
print(f"  IT: {len(df_it)} ({len(df_it)/max(1,len(title_lower))*100:.0f}%)")

### Bước 2: Đọc skills (`job_skills.csv`) — chunked

Vì file 642 MB, dùng `chunksize` để không load hết vào RAM cùng lúc.

In [ ]:
print("\n[2/4] Loading skills...")
skill_map = {}
rows_read = 0
for chunk in pd.read_csv(SKILLS, usecols=["job_link", "job_skills"], chunksize=CHUNK, low_memory=False):
    rows_read += len(chunk)
    sub = chunk[chunk["job_link"].isin(it_links)]
    for _, r in sub.iterrows():
        skill_map[r["job_link"]] = r.get("job_skills", "")
    if rows_read % (CHUNK * 5) == 0:
        print(f"  ...{rows_read/1e6:.1f}M scanned, {len(skill_map)} matched")

del chunk; gc.collect()
print(f"  Skills matched: {len(skill_map)}")

### Bước 3: Đọc summary (`job_summary.csv`) — csv.DictReader

File 4.9 GB, **không thể** dùng `pd.read_csv` (sẽ tràn RAM).
Dùng `csv.DictReader` (implement bằng C, đọc từng dòng) để quét.

Chỉ lưu `job_link` + `job_summary` cho các job IT.

In [ ]:
print("\n[3/4] Loading summary (csv.DictReader)...")
sum_map = {}
rows_read = 0
with open(SUMMARY, "r", encoding="utf-8", errors="replace") as f:
    reader = csv.DictReader(f)
    for row in reader:
        rows_read += 1
        jlink = row.get("job_link", "")
        if jlink in it_links:
            sum_map[jlink] = row.get("job_summary", "")
        if rows_read % 2_000_000 == 0:
            print(f"  ...{rows_read/1e6:.1f}M scanned, {len(sum_map)} matched")

print(f"  Summaries matched: {len(sum_map)}")

### Bước 4: Xây dựng dataset cuối cùng

Với mỗi job IT:
1. Lấy `job_level` + `job_title` → xác định seniority
2. Lấy `job_location` → chuẩn hoá bang
3. Lấy `job_title` → xác định IT domain
4. Lấy `job_type` → chuẩn hoá
5. Gộp `job_summary` + `job_skills` → đếm kỹ năng
6. Trích lương từ `job_summary`

In [ ]:
print("\n[4/4] Building dataset...")
rows = []
for _, post in df_it.iterrows():
    link = post["job_link"]
    title = post.get("job_title", "")
    level = post.get("job_level", "")
    jtype = post.get("job_type", "")
    loc = post.get("job_location", "")
    company = post.get("company", "")

    sk = skill_map.get(link, "")
    sm = sum_map.get(link, "")
    combined = f"{sm} {sk}"

    sen = nsen(level, title)
    st = ns(loc)
    dom = md(title)
    jt = njt(jtype)
    sal = extract_salary(sm)

    feats = {f"skill_{c}": count_skill(combined, kws) for c, kws in SKILL_KW.items()}
    feats["num_skills"] = sum(feats.values())
    feats["skill_diversity"] = sum(1 for v in feats.values() if v > 0)

    rows.append({
        "job_link": link, "job_title": title, "company": company,
        "state": st, "it_domain": dom, "seniority_level": sen, "job_type": jt,
        "salary_annual": sal, **feats,
    })

    if len(rows) % 50_000 == 0:
        print(f"  ...{len(rows)} rows built")

df = pd.DataFrame(rows)
print(f"  Total: {len(df)}")

---
## 5. Hậu xử lý

### 5a. Loại bỏ outlier lương (IQR)

Dùng IQR để loại salary quá thấp (< $15K hoặc Q1 - 1.5*IQR) hoặc quá cao (> $500K hoặc Q3 + 1.5*IQR).
Chỉ áp dụng lên các dòng **có** salary; dòng không có salary giữ nguyên.

In [ ]:
sal = df["salary_annual"].dropna()
print(f"  Salaries before outlier: {len(sal)}")
if len(sal) >= 50:
    q1, q3 = sal.quantile(0.25), sal.quantile(0.75)
    iqr = q3 - q1
    lo, hi = max(q1 - 1.5 * iqr, 15000), min(q3 + 1.5 * iqr, 500000)
    n_before = len(df)
    df = df[(df["salary_annual"].isna()) | ((df["salary_annual"] >= lo) & (df["salary_annual"] <= hi))]
    print(f"  Outliers removed: {n_before - len(df)}")
    print(f"  Salaries after outlier: {df['salary_annual'].notna().sum()}")

### 5b. Sắp xếp cột và xuất file CSV

In [ ]:
cols = [
    "job_link", "job_title", "company", "state", "it_domain",
    "seniority_level", "job_type", "salary_annual",
    "num_skills", "skill_diversity",
    "skill_programming", "skill_cloud", "skill_ai_ml", "skill_database",
    "skill_devops", "skill_framework", "skill_data_engineering",
    "skill_security", "skill_soft_skills",
]
df = df[[c for c in cols if c in df.columns]]

# Fill NaN kỹ năng = 0
sc = [c for c in df.columns if c.startswith("skill_") or c in ("num_skills", "skill_diversity")]
df[sc] = df[sc].fillna(0).astype(int)

df.to_csv(OUTPUT, index=False)

---
## 6. Thống kê đầu ra

In [ ]:
print(f"\n{'='*60}")
print(f"  Output: {OUTPUT}")
print(f"  Rows:   {len(df)}")
print(f"\n  Seniority: {dict(df['seniority_level'].value_counts())}")
print(f"  State (top10): {dict(df['state'].value_counts().head(10))}")
print(f"  Job type: {dict(df['job_type'].value_counts())}")
print(f"  Domain: {dict(df['it_domain'].value_counts())}")
print(f"  Salaries: {df['salary_annual'].notna().sum()}")
print(f"{'='*60}")

---
## Next steps sau khi chạy notebook này

1. **Retrain models**: chạy `retrain_all.py` (hoặc mở `notebooks/model_training.ipynb`)
2. **Chạy server**: `python -m backend.server` → http://localhost:5000
3. **Docker**: `docker compose up --build`

```bash
# Quick sanity check
python -c "
import pandas as pd
df = pd.read_csv('data/it_jobs_processed.csv')
print('Rows:', len(df))
print('Seniority:', dict(df['seniority_level'].value_counts()))
print('Salaries:', df['salary_annual'].notna().sum())
"```